# Long-Horizon Emissions Forecasting for 2030 Target Assessment: A Comparative Study of N-HiTS, XGBoost, and Bayesian Models in Fast-Moving Consumer Goods Supply Chains
**Authors**: Idris Alugo 

**Paper**: *Applied Energy* (submitted 2026)  
**Zenodo DOI**: [to be assigned]

This notebook orchestrates the full pipeline:
1. Data loading & feature engineering  
2. Per-facility model training (N-HiTS, XGBoost-Quantile, BNN)  
3. Out-of-fold meta-feature construction  
4. Stacking ensemble (meta-learner) training  
5. 2030 forecast generation  
6. Risk & compliance assessment  
7. SHAP explainability  
8. Publication figures

All heavy logic lives in `src/` — this notebook is the **single entry point** to reproduce all results.


## 0. Environment

In [ ]:
import sys, importlib
print(f"Python {sys.version}")

required = [
    "numpy", "pandas", "torch", "xgboost",
    "neuralforecast", "shap", "sklearn",
    "matplotlib", "seaborn", "scipy",
]
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg}")
    except ImportError:
        missing.append(pkg)
        print(f"  ✗ {pkg}  ← MISSING")

if missing:
    raise ImportError(f"Install missing packages: pip install {' '.join(missing)}")
else:
    print("\nAll dependencies satisfied.")


## 1. Imports & Logging

In [ ]:
import logging
import warnings
import numpy as np
import pandas as pd

# src modules
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # adjust if running from notebooks/

from config import (
    DATA_PATH, TARGET_COL, TEST_MONTHS, RANDOM_SEED,
    BASELINE_YEAR, TARGET_REDUCTION_RATE, MIN_TRAIN_ROWS,
)
from src.preprocessing import (
    load_and_clean_data,
    create_global_features_with_hierarchy,
)
from src.models.nhits_model   import run_nhits, predict_nhits_2030, shap_nhits
from src.models.xgboost_model import run_xgboost_quantile, predict_xgboost_2030, shap_xgboost
from src.models.bnn_model     import run_bnn, mc_predict_bnn, shap_bnn
from src.models.meta_learner  import (
    build_meta_features, train_meta_learner,
    predict_ensemble, evaluate_meta_learner,
)
from src.evaluation import (
    prepare_facility_targets,
    compute_calibrated_probabilities,
    compute_stakeholder_metrics,
    compare_model_metrics,
    compute_regional_summary,
    aggregate_shap_records,
    compute_portfolio_probability,
)
from src.visualization import plot_all

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("pipeline")
np.random.seed(RANDOM_SEED)
logger.info("Imports complete.")

## 2. Data Loading & Feature Engineering

In [ ]:
# Load and clean raw data
df_raw = load_and_clean_data(DATA_PATH)
logger.info("Raw data: %s", df_raw.shape)

# Define base numeric features (before hierarchy/lag enrichment)
BASE_NUMERIC_FEATURES = [
    "Month", "Year", "DayOfYear",
    "Production", "Energy_MWh", "Waste_Kg",
    "Renewable_percent", "PolicyWeight",
]
region_cols = [c for c in df_raw.columns if c.startswith("Region_")]
base_features = BASE_NUMERIC_FEATURES + region_cols

# Create enhanced feature set (lags, rolling stats, hierarchy embeddings)
df, global_scalers, features = create_global_features_with_hierarchy(df_raw, base_features)

logger.info("Enhanced data: %s | Features: %d", df.shape, len(features))
logger.info("Facilities: %s", df["Facility"].nunique())
df.head(3)

## 2b. Facility Emissions Profile Clustering


In [ ]:
import os
from src.models.clustering_model import run_facility_clustering
from src.visualization import plot_facility_clusters

clust = run_facility_clustering(df_raw)

print("-- Silhouette Scores --")
for k, s in sorted(clust["sil_scores"].items()):
    marker = "  <-- selected (elbow)" if k == clust["best_k"] else ""
    print(f"  K={k}  silhouette={s:.4f}{marker}")

print("\n-- Cluster Membership (K=3, manuscript-aligned) --")
n_fac = len(clust["fac_df"])
for label, members in [
    ("High Emitter",   clust["high_facs"]),
    ("Medium Emitter", clust["medium_facs"]),
    ("Low Emitter",    clust["low_facs"]),
]:
    pct    = round(len(members) / n_fac * 100)
    mean_e = clust["fac_df"].loc[members, "MeanEmissions"].mean()
    print(f"  {label}: {len(members)} ({pct}%)  mean={mean_e:.1f} tCO2  -> {', '.join(members)}")

os.makedirs("../figures", exist_ok=True)
cluster_fig_path = plot_facility_clusters(
    clust["cluster_df"], clust["sil_scores"], clust["best_k"],
    output_dir="../figures", show=True,
)
print(f"\nSaved: {cluster_fig_path}")


## 3. Per-Facility Model Training

> Trains N-HiTS, XGBoost-Quantile, and BNN for each facility. Collects OOF predictions for meta-learning.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

results          = []
meta_feature_records = []
shap_records     = []
SHAP_FEATURES    = ["Energy_MWh", "Production", "EmissionsLag1", "EmissionsLag3"]
shap_raw_rows    = {"nhits": [], "xgb": [], "bnn": []}
X_raw_rows       = {"nhits": [], "xgb": [], "bnn": []}
bnn_res_dict     = {}
xgb_res_dict     = {}
nhits_res_dict   = {}

tscv = TimeSeriesSplit(n_splits=3)

for fac in df["Facility"].unique():
    df_fac = df[df["Facility"] == fac].sort_values("Date").reset_index(drop=True)
    logger.info("=" * 60)
    logger.info("Facility: %s  (%d rows)", fac, len(df_fac))

    if len(df_fac) < MIN_TRAIN_ROWS or df_fac[TARGET_COL].nunique() < 2:
        logger.warning("Skipping %s — insufficient data or no target variance.", fac)
        continue

    nhits_oof = np.full(len(df_fac), np.nan)
    xgb_oof   = np.full(len(df_fac), np.nan)
    bnn_oof   = np.full(len(df_fac), np.nan)

    # ── Out-of-fold predictions (TimeSeriesSplit) ──────────────────────────
    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(df_fac)):
        df_train = df_fac.iloc[train_idx]
        df_test  = df_fac.iloc[test_idx]
        n_test   = len(df_test)

        # N-HiTS OOF
        try:
            res = run_nhits(
                pd.concat([df_train, df_test], ignore_index=True),
                fac, features, TARGET_COL, test_months=n_test,
            )
            if res and res.get("model") is not None:
                for i, idx in enumerate(test_idx):
                    if i < len(res["y_pred"]):
                        nhits_oof[idx] = res["y_pred"][i]
        except Exception as e:
            logger.debug("N-HiTS OOF fold %d failed for %s: %s", fold_idx, fac, e)

        # XGBoost OOF
        try:
            res = run_xgboost_quantile(
                pd.concat([df_train, df_test], ignore_index=True),
                fac, features, TARGET_COL, test_months=n_test,
            )
            if res and res.get("models") and 0.5 in res["models"]:
                preds = res["models"][0.5].predict(df_test[features].values)
                for i, idx in enumerate(test_idx):
                    if i < len(preds):
                        xgb_oof[idx] = preds[i]
        except Exception as e:
            logger.debug("XGBoost OOF fold %d failed for %s: %s", fold_idx, fac, e)

        # BNN OOF
        try:
            res = run_bnn(
                pd.concat([df_train, df_test], ignore_index=True),
                fac, features, TARGET_COL, test_months=n_test, n_epochs=100,
            )
            if res and res.get("model") is not None:
                mc = mc_predict_bnn(res["model"], df_test[features].values)
                preds = mc.mean(axis=0)
                for i, idx in enumerate(test_idx):
                    if i < len(preds):
                        bnn_oof[idx] = preds[i]
        except Exception as e:
            logger.debug("BNN OOF fold %d failed for %s: %s", fold_idx, fac, e)

    # ── Final models trained on full facility data ─────────────────────────
    nhits_res = run_nhits(df_fac, fac, features, TARGET_COL, TEST_MONTHS)
    xgb_res   = run_xgboost_quantile(df_fac, fac, features, TARGET_COL, TEST_MONTHS)
    bnn_res   = run_bnn(df_fac, fac, features, TARGET_COL, TEST_MONTHS, n_epochs=300)

    nhits_res_dict[fac] = nhits_res
    xgb_res_dict[fac]   = xgb_res
    bnn_res_dict[fac]   = bnn_res

    results.append({
        "Facility":    fac,
        "NHITSMAPE":   nhits_res.get("mape", np.nan),
        "NHITSRMSE":   nhits_res.get("rmse", np.nan),
        "XGBoostMAPE": xgb_res.get("mape", np.nan),
        "XGBoostRMSE": xgb_res.get("rmse", np.nan),
        "BNNMAPE":     bnn_res.get("mape", np.nan),
        "BNNRMSE":     bnn_res.get("rmse", np.nan),
    })

    # ── Collect OOF meta-feature rows ─────────────────────────────────────
    for i in range(len(df_fac)):
        has_nhits = not np.isnan(nhits_oof[i])
        has_xgb   = not np.isnan(xgb_oof[i])
        has_bnn   = not np.isnan(bnn_oof[i])
        if has_nhits or has_xgb or has_bnn:
            meta_feature_records.append({
                "Facility": fac,
                "NHITS_OOF": nhits_oof[i] if has_nhits else 0.0,
                "XGB_OOF":   xgb_oof[i]   if has_xgb   else 0.0,
                "BNN_OOF":   bnn_oof[i]   if has_bnn   else 0.0,
                "y_true":    df_fac.iloc[i][TARGET_COL],
            })

    # ── SHAP analysis ──────────────────────────────────────────────────────
    X_val = df_fac[features].values[-TEST_MONTHS:]
    X_train_shap = df_fac[features].values[:-TEST_MONTHS]

    mean_abs_xgb = mean_abs_bnn = mean_abs_nhits = None
    _feat_idx = [features.index(f) for f in SHAP_FEATURES if f in features]

    if xgb_res and xgb_res.get("models") and 0.5 in xgb_res["models"]:
        try:
            sv = shap_xgboost(xgb_res["models"][0.5], X_val, features, fac, show_plot=False)
            if sv is not None:
                mean_abs_xgb = np.abs(sv).mean(axis=0)
                for _i in range(len(sv)):
                    _sr = {"Facility": fac}
                    _xr = {"Facility": fac}
                    for _j, _fn in zip(_feat_idx, SHAP_FEATURES):
                        _sr[_fn] = float(sv[_i, _j])
                        _xr[_fn] = float(X_val[_i, _j])
                    shap_raw_rows["xgb"].append(_sr)
                    X_raw_rows["xgb"].append(_xr)
        except Exception as e:
            logger.debug("XGB SHAP failed for %s: %s", fac, e)

    if bnn_res and bnn_res.get("model") is not None:
        try:
            sv = shap_bnn(bnn_res["model"], X_train_shap, X_val, features, fac, show_plot=False)
            if sv is not None:
                mean_abs_bnn = np.abs(sv).mean(axis=0)
                for _i in range(len(sv)):
                    _sr = {"Facility": fac}
                    _xr = {"Facility": fac}
                    for _j, _fn in zip(_feat_idx, SHAP_FEATURES):
                        _sr[_fn] = float(sv[_i, _j])
                        _xr[_fn] = float(X_val[_i, _j])
                    shap_raw_rows["bnn"].append(_sr)
                    X_raw_rows["bnn"].append(_xr)
        except Exception as e:
            logger.debug("BNN SHAP failed for %s: %s", fac, e)

    try:
        sv = shap_nhits(X_train_shap, X_val, features, fac, show_plot=False)
        if sv is not None:
            mean_abs_nhits = np.abs(sv).mean(axis=0)
            for _i in range(len(sv)):
                _sr = {"Facility": fac}
                _xr = {"Facility": fac}
                for _j, _fn in zip(_feat_idx, SHAP_FEATURES):
                    _sr[_fn] = float(sv[_i, _j])
                    _xr[_fn] = float(X_val[_i, _j])
                shap_raw_rows["nhits"].append(_sr)
                X_raw_rows["nhits"].append(_xr)
    except Exception as e:
        logger.debug("N-HiTS SHAP failed for %s: %s", fac, e)

    if any(x is not None for x in [mean_abs_xgb, mean_abs_bnn, mean_abs_nhits]):
        zeros = np.zeros(len(features))
        for i, feat in enumerate(features):
            shap_records.append({
                "Facility":        fac,
                "Feature":         feat,
                "MeanAbsSHAPNHITS": mean_abs_nhits[i] if mean_abs_nhits is not None else zeros[i],
                "MeanAbsSHAPXGB":   mean_abs_xgb[i] if mean_abs_xgb is not None else zeros[i],
                "MeanAbsSHAPBNN":   mean_abs_bnn[i] if mean_abs_bnn is not None else zeros[i],
            })

results_df = pd.DataFrame(results)
logger.info("Training complete. %d facilities.", len(results_df))
results_df[["Facility","NHITSMAPE","XGBoostMAPE","BNNMAPE"]]

## 4. Meta-Learner Ensemble Training

In [ ]:
import pandas as pd

if "meta_feature_records" not in globals():
    raise RuntimeError("Run Cell 8 (Per-Facility Model Training) before this cell.")

meta_df = pd.DataFrame(meta_feature_records)
logger.info("Meta-feature rows collected: %d", len(meta_df))

# Augment with uncertainty & agreement features
meta_df_enhanced = build_meta_features(meta_df, bnn_res_dict, xgb_res_dict)

# Train stacking ensemble
meta_models, meta_feature_cols, meta_weights = train_meta_learner(
    meta_df_enhanced, verbose=True
)
logger.info("Meta-learner weights: %s", meta_weights)

# OOF evaluation
oof_eval = evaluate_meta_learner(
    meta_models, meta_weights, meta_feature_cols,
    meta_df_enhanced,
    individual_oof_cols={"N-HiTS": "NHITS_OOF", "XGBoost": "XGB_OOF", "BNN": "BNN_OOF"},
)
logger.info("Ensemble OOF MAPE=%.2f%%  RMSE=%.4f", oof_eval["ensemble_mape"], oof_eval["ensemble_rmse"])
pd.DataFrame([oof_eval]).T


## 5. 2030 Forecast Generation

In [ ]:
from src.preprocessing import prepare_2030_features

nhits_quantile_preds = {}
xgb_quantile_preds   = {}
bnn_mc_preds         = {}

for fac in results_df["Facility"].unique():
    df_fac = df[df["Facility"] == fac].copy()
    X_2030 = prepare_2030_features(df, fac, features, global_scalers, base_features)

    # N-HiTS
    nhits_res = nhits_res_dict.get(fac)
    if nhits_res and nhits_res.get("model") is not None:
        preds = predict_nhits_2030(nhits_res["model"], df_fac, fac, features)
        if preds:
            nhits_quantile_preds[fac] = preds

    # XGBoost
    xgb_res = xgb_res_dict.get(fac)
    if xgb_res and xgb_res.get("models"):
        preds = predict_xgboost_2030(xgb_res["models"], X_2030)
        if preds:
            xgb_quantile_preds[fac] = preds

    # BNN
    bnn_res = bnn_res_dict.get(fac)
    if bnn_res and bnn_res.get("model") is not None:
        samples = mc_predict_bnn(bnn_res["model"], X_2030)
        samples = np.clip(samples, 0, None)
        bnn_mc_preds[fac] = samples.flatten()

logger.info(
    "2030 forecasts: N-HiTS=%d  XGBoost=%d  BNN=%d",
    len(nhits_quantile_preds), len(xgb_quantile_preds), len(bnn_mc_preds),
)


## 6. Risk & Compliance Assessment

In [ ]:
# Facility-level baselines and targets
facility_baseline = prepare_facility_targets(
    df_raw,
    baseline_year=BASELINE_YEAR,
    target_reduction=TARGET_REDUCTION_RATE,
)

# Per-facility compliance probabilities
risk_df = compute_calibrated_probabilities(
    facility_baseline,
    nhits_quantile_preds,
    xgb_quantile_preds,
    bnn_mc_preds,
    results_df,
    n_bootstrap=1000,
)

# Regional roll-up
regional_summary = compute_regional_summary(risk_df, facility_baseline)

# Model comparison
comparison_df = compare_model_metrics(results_df)

print("\n── Risk Distribution ──────────────────────────")
print(risk_df["RiskLevelEnsemble"].value_counts().to_string())
print("\n── Model Comparison ───────────────────────────")
print(comparison_df.to_string())

# Build forecasts_df from in-memory quantile dicts for portfolio bootstrap
forecast_rows_port = []
for fac in sorted(set(nhits_quantile_preds) | set(xgb_quantile_preds) | set(bnn_mc_preds)):
    nhits_p = nhits_quantile_preds.get(fac, {})
    xgb_p   = xgb_quantile_preds.get(fac, {})
    bnn_p   = bnn_mc_preds.get(fac)
    bnn_q10 = float(np.percentile(bnn_p, 10)) if bnn_p is not None else float("nan")
    bnn_q50 = float(np.percentile(bnn_p, 50)) if bnn_p is not None else float("nan")
    bnn_q90 = float(np.percentile(bnn_p, 90)) if bnn_p is not None else float("nan")
    forecast_rows_port.append({
        "Facility": fac,
        "NHITS_Q10": nhits_p.get(0.1, float("nan")), "NHITS_Q50": nhits_p.get(0.5, float("nan")), "NHITS_Q90": nhits_p.get(0.9, float("nan")),
        "XGB_Q10":   xgb_p.get(0.1, float("nan")),  "XGB_Q50":  xgb_p.get(0.5, float("nan")),  "XGB_Q90":  xgb_p.get(0.9, float("nan")),
        "BNN_Q10":   bnn_q10, "BNN_Q50": bnn_q50, "BNN_Q90": bnn_q90,
    })
forecasts_df_port = pd.DataFrame(forecast_rows_port)

portfolio_metrics = compute_portfolio_probability(
    risk_df, forecasts_df=forecasts_df_port, results_df=results_df,
    n_bootstrap=10_000_000,
)

print(f"\n── Portfolio Compliance ──────────────────────────────")
print(f"  P_portfolio (all 20 meet target)  = {portfolio_metrics['P_portfolio_pct']}")
print(f"  E[compliant facilities]           = {portfolio_metrics['E_compliant']:.2f} ± {portfolio_metrics['E_compliant_std']:.2f}")
print(f"  (out of {portfolio_metrics['N_facilities']} facilities, N={portfolio_metrics['N_bootstrap']:,} samples)")

# Uncertainty variance decomposition (Eq. 8)
_MAPE_GATE = 80.0
_decomp_rows = []
for _, _r in risk_df.iterrows():
    _fac = _r["Facility"]
    _a   = results_df[results_df["Facility"] == _fac]
    if _a.empty: continue
    _mapes = {"nhits": _a["NHITSMAPE"].values[0], "xgb": _a["XGBoostMAPE"].values[0], "bnn": _a["BNNMAPE"].values[0]}
    _means = {"nhits": _r["PredictionNHITS"], "xgb": _r["PredictionXGBoost"], "bnn": _r["PredictionBNN"]}
    _stds  = {"nhits": abs(_r["UncertaintyNHITS"]), "xgb": abs(_r["UncertaintyXGBoost"]), "bnn": abs(_r["UncertaintyBNN"])}
    _mods  = [k for k in ["nhits","xgb","bnn"] if not np.isnan(_mapes[k]) and _mapes[k] <= _MAPE_GATE and not np.isnan(_means[k]) and not np.isnan(_stds[k])]
    if not _mods: _mods = [k for k in ["nhits","xgb","bnn"] if not np.isnan(_means[k]) and not np.isnan(_stds[k])]
    if not _mods: continue
    _rw = np.array([1.0/(_mapes[k]+1.0) for k in _mods]); _w = _rw/_rw.sum()
    _mu = np.array([_means[k] for k in _mods]); _sig = np.array([_stds[k] for k in _mods])
    _mu_bar = float(np.dot(_w, _mu))
    _within = float(np.dot(_w, _sig**2)); _between = float(np.dot(_w, (_mu - _mu_bar)**2)); _total = _within + _between
    _decomp_rows.append({"Facility": _fac, "within_model_var": round(_within,4), "between_model_var": round(_between,4),
                          "total_var": round(_total,4), "within_pct": round(_within/_total*100,2) if _total>0 else float("nan"),
                          "between_pct": round(_between/_total*100,2) if _total>0 else float("nan")})
decomp_df = pd.DataFrame(_decomp_rows).sort_values("Facility")
logger.info("Variance decomposition: %d facilities", len(decomp_df))

## 7. SHAP Feature Importance

In [ ]:
shap_summary = aggregate_shap_records(shap_records, features)

print("── Top 15 Features by Average SHAP Contribution ──")
print(
    shap_summary.head(15)[
        ["Feature", "NHITSContribution", "XGBContribution",
         "BNNContribution", "AvgContribution", "Cumulative"]
    ].to_string(index=False)
)


## 8. Publication Figures

In [ ]:
# Build shap_raw / X_raw DataFrames from the training loop accumulators
shap_raw = {m: pd.DataFrame(shap_raw_rows[m]) for m in ["nhits", "xgb", "bnn"]}
X_raw    = {m: pd.DataFrame(X_raw_rows[m])    for m in ["nhits", "xgb", "bnn"]}

saved_paths = plot_all(
    results_df    = results_df,
    risk_df       = risk_df,
    comparison_df = comparison_df,
    shap_summary  = shap_summary,
    output_dir    = "../figures",
    show          = True,          # set False for headless runs
    shap_raw      = shap_raw,
    X_raw         = X_raw,
    decomp_df     = decomp_df,
)
for p in saved_paths:
    print("Saved:", p)


## 9. Export Results

In [ ]:
os.makedirs("../results", exist_ok=True)

results_df.to_csv("../results/model_accuracy.csv",      index=False)
risk_df.to_csv("../results/risk_assessment.csv",         index=False)
regional_summary.to_csv("../results/regional_summary.csv", index=False)
comparison_df.to_csv("../results/model_comparison.csv")

if len(shap_summary) > 0:
    shap_summary.to_csv("../results/shap_summary.csv", index=False)

# 2030 per-facility forecast quantiles
forecast_rows = []
for fac in sorted(set(nhits_quantile_preds) | set(xgb_quantile_preds) | set(bnn_mc_preds)):
    nhits_p = nhits_quantile_preds.get(fac, {})
    xgb_p   = xgb_quantile_preds.get(fac, {})
    bnn_p   = bnn_mc_preds.get(fac)
    forecast_rows.append({
        "Facility":  fac,
        "NHITS_Q10": nhits_p.get(0.1, float("nan")),
        "NHITS_Q50": nhits_p.get(0.5, float("nan")),
        "NHITS_Q90": nhits_p.get(0.9, float("nan")),
        "XGB_Q10":   xgb_p.get(0.1,  float("nan")),
        "XGB_Q50":   xgb_p.get(0.5,  float("nan")),
        "XGB_Q90":   xgb_p.get(0.9,  float("nan")),
        "BNN_Q10":   float(np.percentile(bnn_p, 10)) if bnn_p is not None else float("nan"),
        "BNN_Q50":   float(np.percentile(bnn_p, 50)) if bnn_p is not None else float("nan"),
        "BNN_Q90":   float(np.percentile(bnn_p, 90)) if bnn_p is not None else float("nan"),
    })
pd.DataFrame(forecast_rows).to_csv("../results/forecasts_2030.csv", index=False)


# Portfolio summary
pd.DataFrame([{
    'P_portfolio':     portfolio_metrics['P_portfolio'],
    'P_portfolio_pct': portfolio_metrics['P_portfolio_pct'],
    'E_compliant':     portfolio_metrics['E_compliant'],
    'E_compliant_std': portfolio_metrics['E_compliant_std'],
    'N_facilities':    portfolio_metrics['N_facilities'],
    'N_bootstrap':     portfolio_metrics['N_bootstrap'],
}]).to_csv('../results/portfolio_summary.csv', index=False)
portfolio_metrics['per_facility'].to_csv('../results/portfolio_per_facility.csv', index=False)

# Raw SHAP matrices for dependence plots
os.makedirs('../results/shap_raw', exist_ok=True)
for _m in ['nhits', 'xgb', 'bnn']:
    shap_raw[_m].to_csv(f'../results/shap_raw/shap_{_m}.csv', index=False)
    X_raw[_m].to_csv(f'../results/shap_raw/X_{_m}.csv', index=False)
logger.info('Raw SHAP matrices saved to ../results/shap_raw/')
decomp_df.to_csv('../results/uncertainty_decomposition.csv', index=False)
logger.info("All results exported to ../results/")
print("Done. Results written to ../results/")

## 8. Long-Horizon Evaluation & Diebold-Mariano Test

### 8.1 Step-wise Horizon Accuracy

To assess how forecast accuracy degrades with forecast horizon — essential for
evaluating 2030 compliance confidence — we use a **single-origin, step-wise** approach:

| Model | Training window | Test window | Forecast steps |
|-------|----------------|-------------|----------------|
| XGBoost | Jan 2022 – Dec 2023 | Jan 2024 – Aug 2025 | h = 1 … 20 |
| BNN | Jan 2022 – Dec 2023 | Jan 2024 – Aug 2025 | h = 1 … 20 |
| N-HiTS (pass 1) | Jan 2022 – Dec 2023 | Jan 2024 – Dec 2024 | h = 1 … 12 |
| N-HiTS (pass 2) | Jan 2022 – Dec 2024 | Jan 2025 – Aug 2025 | h = 13 … 20 |

N-HiTS requires two passes because its horizon is internally capped at 12;
the second pass uses additional training data (noted as a limitation).

### 8.2 Diebold-Mariano Test

Pairwise **Diebold-Mariano (DM) tests** with Harvey-Leybourne-Newbold small-sample
correction assess whether differences in predictive accuracy are statistically
significant. Loss differentials are computed on pooled per-facility × per-step
absolute percentage errors (APE).

H₀: equal predictive accuracy (two-sided, α = 0.05).


In [ ]:
import sys
# Reload src modules so kernel picks up any edits made after it started
for _mod in list(sys.modules.keys()):
    if _mod.startswith('src.'):
        del sys.modules[_mod]

import pandas as pd, numpy as np, os, matplotlib.pyplot as plt
from src.preprocessing import (
    load_data, fill_missing_covariates, remove_outlier_facilities,
    filter_date_range, add_time_features, add_region_dummies,
    create_global_features,
)
from src.evaluation import evaluate_per_step_horizons, run_diebold_mariano_all_pairs
from config import TARGET_COL

# ── 1. Build extended dataset (Jan 2022 – Aug 2025) ──────────────────────
print('Building extended dataset (2022–2025)...')
df_ext_raw = load_data('../data/esgdata.csv')
df_ext_raw = fill_missing_covariates(df_ext_raw)
df_ext_raw = remove_outlier_facilities(df_ext_raw)
df_ext_raw = filter_date_range(df_ext_raw, year_min=2016, year_max=2025)
df_ext_raw = add_time_features(df_ext_raw)
df_ext_raw = add_region_dummies(df_ext_raw)

features_base_ext = ['Energy_MWh', 'Production', 'Renewable_percent',
                     'Month', 'Year', 'PolicyWeight']
df_ext, _, features_ext = create_global_features(df_ext_raw, features_base_ext)

_facilities_ext = sorted(df_ext['Facility'].unique())
print(f'Extended dataset: {len(df_ext)} rows, {len(_facilities_ext)} facilities')
print(f'Date range: {df_ext["Date"].min().date()} → {df_ext["Date"].max().date()}')
print()

# ── 2. Step-wise horizon evaluation ──────────────────────────────────────
print('Running step-wise horizon evaluation (h=1-20)...')
print('  XGBoost & BNN: single origin Dec-2023, test_months=20')
print('  N-HiTS: two passes (h=1-12 from Dec-2023; h=13-20 from Dec-2024)')
print()
step_df, summary_df = evaluate_per_step_horizons(
    df_ext, _facilities_ext, features_ext, metric=TARGET_COL
)

print('=== Step-wise MAPE by Model & Horizon ===')
pivot = summary_df.pivot(index='h', columns='Model', values='MAPE').round(1)
print(pivot.to_string())
print()

# ── 3. Horizon bucket summary ─────────────────────────────────────────────
def _bucket(h):
    if h <= 6:  return 'Short (h=1-6)'
    if h <= 12: return 'Medium (h=7-12)'
    return 'Long (h=13-20)'

step_df['Bucket'] = step_df['h'].apply(_bucket)
bucket_summary = (
    step_df.groupby(['Model', 'Bucket'])['APE']
    .agg(MAPE='mean', Std='std', N='count')
    .reset_index()
)
# Order buckets
bucket_order = ['Short (h=1-6)', 'Medium (h=7-12)', 'Long (h=13-20)']
bucket_summary['Bucket'] = pd.Categorical(bucket_summary['Bucket'], bucket_order, ordered=True)
bucket_summary = bucket_summary.sort_values(['Model', 'Bucket'])
print('=== Bucket Summary ===')
print(bucket_summary.to_string(index=False))
print()

# ── 4. Diebold-Mariano pairwise tests ─────────────────────────────────────
print('Running Diebold-Mariano tests (pooled APE, all pairs)...')
dm_results = run_diebold_mariano_all_pairs(step_df, h=1)
print()
print('=== Diebold-Mariano Test Results ===')
print(dm_results.to_string(index=False))
print()

# ── 5. Plot: MAPE vs forecast horizon ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: step-wise MAPE per model
colors = {'XGBoost': '#2196F3', 'N-HiTS': '#4CAF50', 'BNN': '#FF9800'}
for model, grp in summary_df.groupby('Model'):
    axes[0].plot(grp['h'], grp['MAPE'], marker='o', label=model,
                 color=colors.get(model, 'gray'), linewidth=2, markersize=5)
    axes[0].fill_between(
        grp['h'],
        (grp['MAPE'] - grp['Std_APE']).clip(0),
        grp['MAPE'] + grp['Std_APE'],
        alpha=0.15, color=colors.get(model, 'gray')
    )
axes[0].axvline(12.5, color='red', linestyle='--', linewidth=1, label='2024/2025 boundary')
axes[0].set_xlabel('Forecast horizon h (months)', fontsize=12)
axes[0].set_ylabel('Mean APE (%)', fontsize=12)
axes[0].set_title('Forecast accuracy vs horizon', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: bucket bar chart
import matplotlib.patches as mpatches
model_order = ['XGBoost', 'N-HiTS', 'BNN']
x = np.arange(len(bucket_order))
width = 0.25
for j, model in enumerate(model_order):
    sub = bucket_summary[bucket_summary['Model'] == model].set_index('Bucket')
    vals = [sub.loc[b, 'MAPE'] if b in sub.index else np.nan for b in bucket_order]
    stds = [sub.loc[b, 'Std']  if b in sub.index else 0.0   for b in bucket_order]
    axes[1].bar(x + j * width, vals, width, label=model,
                color=colors.get(model, 'gray'), alpha=0.85)
    axes[1].errorbar(x + j * width, vals, yerr=stds, fmt='none',
                     color='black', capsize=4, linewidth=1.5)
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(bucket_order, fontsize=10)
axes[1].set_ylabel('Mean APE (%)', fontsize=12)
axes[1].set_title('Accuracy by horizon bucket', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/horizon_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/horizon_accuracy.png')

# ── 6. Save CSVs ──────────────────────────────────────────────────────────
step_df.to_csv('../results/horizon_step_df.csv', index=False)
summary_df.to_csv('../results/horizon_summary.csv', index=False)
bucket_summary.to_csv('../results/horizon_bucket_summary.csv', index=False)
dm_results.to_csv('../results/dm_test_results.csv', index=False)
print('Saved results/horizon_step_df.csv')
print('Saved results/horizon_summary.csv')
print('Saved results/horizon_bucket_summary.csv')
print('Saved results/dm_test_results.csv')


## 9. Facility Data-Quality Report

Checks completeness and missingness for every facility in the **raw**
dataset (before outlier removal), then for the **retained** 20 facilities.
Use this to verify that included facilities meet quality thresholds.


In [ ]:
from src.preprocessing import compute_facility_quality_report, load_data

# Quality report on raw data (all facilities, no filtering)
df_raw_all = load_data('../data/esgdata.csv')
quality_all = compute_facility_quality_report(df_raw_all)

# Quality report on retained facilities only (those in the main pipeline df)
_retained_facs = sorted(df['Facility'].unique())
quality_retained = compute_facility_quality_report(
    df_raw_all[df_raw_all['Facility'].isin(_retained_facs)]
)

print('=== All facilities ===')
print(quality_all[['Facility','N_rows','Completeness_pct','Target_missing_pct','Quality_tier']].to_string(index=False))
print(f"\nTier counts (all): {quality_all['Quality_tier'].value_counts().to_dict()}")

print('\n=== Retained facilities ===')
print(quality_retained[['Facility','N_rows','Date_min','Date_max','Completeness_pct','Quality_tier']].to_string(index=False))
print(f"\nTier counts (retained): {quality_retained['Quality_tier'].value_counts().to_dict()}")

quality_all.to_csv('../results/facility_quality_report_all.csv', index=False)
quality_retained.to_csv('../results/facility_quality_report_retained.csv', index=False)
print('\nSaved results/facility_quality_report_all.csv')
print('Saved results/facility_quality_report_retained.csv')


## 10. Temporal Robustness Check (2025 Holdout)

Trains fresh model instances on **Jan 2022 – Dec 2024** (36 months) and
evaluates on **Jan–Aug 2025** (8 months) — data never seen by any model
in the main pipeline.

**Purpose:** Verify that model accuracy rankings are stable on genuinely
future data, guarding against overfitting to the 2024 test year.

> This cell is self-contained. It does **not** modify `results_df`,
> `risk_df`, or any figure. Outputs go to `results/robustness_2025.csv`.


In [ ]:
import pandas as pd, numpy as np, os
from src.preprocessing import (
    load_data, fill_missing_covariates, remove_outlier_facilities,
    filter_date_range, add_time_features, add_region_dummies,
    create_global_features,
)
from src.models.xgboost_model import run_xgboost_quantile
from src.models.bnn_model     import run_bnn
from src.models.nhits_model   import run_nhits
from config import MIN_TRAIN_ROWS, TARGET_COL

# -- 1. Build dataset extended to Aug 2025 ---------------------------------
ROBUSTNESS_MAX_YEAR = 2025
ROBUSTNESS_TEST_MONTHS = 8  # Jan-Aug 2025

df_rob_raw = load_data('../data/esgdata.csv')
df_rob_raw = fill_missing_covariates(df_rob_raw)
df_rob_raw = remove_outlier_facilities(df_rob_raw)
df_rob_raw = filter_date_range(df_rob_raw, year_min=2016, year_max=ROBUSTNESS_MAX_YEAR)
df_rob_raw = add_time_features(df_rob_raw)
df_rob_raw = add_region_dummies(df_rob_raw)

features_base_rob = ['Energy_MWh', 'Production', 'Renewable_percent',
                      'Month', 'Year', 'PolicyWeight']
df_rob, _, features_rob = create_global_features(df_rob_raw, features_base_rob)

# Use TARGET_COL from config (avoids hardcoding column name)
_target_col = TARGET_COL
print(f'Target column: {_target_col}')

facilities_rob = sorted(df_rob['Facility'].unique())
print(f'Robustness dataset: {len(df_rob)} rows, {len(facilities_rob)} facilities')
print(f'Date range: {df_rob["Date"].min().date()} to {df_rob["Date"].max().date()}')
print(f'Train: all months except last {ROBUSTNESS_TEST_MONTHS}  |  Test: last {ROBUSTNESS_TEST_MONTHS} months (Jan-Aug 2025)')
print()

# -- 2. Train & evaluate each facility -------------------------------------
rob_records = []
for fac in facilities_rob:
    df_fac = df_rob[df_rob['Facility'] == fac].dropna(subset=[_target_col])
    if len(df_fac) - ROBUSTNESS_TEST_MONTHS < MIN_TRAIN_ROWS:
        print(f'  SKIP {fac}: insufficient train rows ({len(df_fac)} total)')
        continue

    xgb_res   = run_xgboost_quantile(df_rob, fac, features_rob, test_months=ROBUSTNESS_TEST_MONTHS)
    bnn_res   = run_bnn(df_rob, fac, features_rob, test_months=ROBUSTNESS_TEST_MONTHS)
    nhits_res = run_nhits(df_rob, fac, feature_cols=features_rob, test_months=ROBUSTNESS_TEST_MONTHS)

    rob_records.append({
        'Facility':    fac,
        'XGBoostMAPE': xgb_res.get('mape', np.nan),
        'XGBoostRMSE': xgb_res.get('rmse', np.nan),
        'BNNMAPE':     bnn_res.get('mape', np.nan),
        'BNNRMSE':     bnn_res.get('rmse', np.nan),
        'NHITSMAPE':   nhits_res.get('mape', np.nan),
        'NHITSRMSE':   nhits_res.get('rmse', np.nan),
    })
    print(f'  {fac}  XGB={xgb_res.get("mape", float("nan")):.1f}%  BNN={bnn_res.get("mape", float("nan")):.1f}%  NHiTS={nhits_res.get("mape", float("nan")):.1f}%')

rob_df = pd.DataFrame(rob_records)

# -- 3. Summary comparison -------------------------------------------------
print()
print('=== Robustness Summary (Train 2022-2024, Test Jan-Aug 2025) ===')
rob_summary_rows = []
for model, mape_col, rmse_col in [
    ('N-HiTS',  'NHITSMAPE',  'NHITSRMSE'),
    ('XGBoost', 'XGBoostMAPE','XGBoostRMSE'),
    ('BNN',     'BNNMAPE',    'BNNRMSE'),
]:
    vals = rob_df[mape_col].dropna()
    rob_summary_rows.append({
        'Model': model,
        'Mean MAPE (%)': round(vals.mean(), 2),
        'Std MAPE (%)':  round(vals.std(),  2),
        'N': len(vals),
    })
rob_summary = pd.DataFrame(rob_summary_rows).set_index('Model')
print(rob_summary.to_string())

print()
print('=== Primary results (Train 2022-2023, Test Jan-Dec 2024) for comparison ===')
primary_summary = comparison_df.reset_index()[['Model','Mean MAPE (%)','Std MAPE (%)','N Facilities']]
print(primary_summary.to_string(index=False))

# -- 4. Save ---------------------------------------------------------------
os.makedirs('../results', exist_ok=True)
rob_df.to_csv('../results/robustness_2025.csv', index=False)
rob_summary.to_csv('../results/robustness_2025_summary.csv')
print()
print('Saved results/robustness_2025.csv')
print('Saved results/robustness_2025_summary.csv')
